# Siem Reap Soil Parameters — 01 Exploratory Data Analysis

The borehole file **exactly as it is**, before any rule is applied: what is measured, where, at which depth, in which soil, and whether the values agree with each other. Nothing is filtered or saved for the models here (that is notebook 02).

**What you'll learn:**
- which columns the file has and how complete each one is
- how many values of each soil parameter exist per depth band and per borehole (the data status)
- how the parameters change with depth and between Clay and Sand
- whether the lab values are consistent (plasticity chart, phase relations)
- which parameters are usable for spatial regression

| | |
|---|---|
| Input | `data/raw/SiemReap_BH_Master_Downtown_DEM_Region_v1_ArcGIS.csv` (fingerprint checked) |
| Output | tables and figures in `outputs/01_eda/`; the borehole map in `outputs/01_eda/private/` |
| Run time | under 1 minute |

## 1. Import Libraries

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import utils as U

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
SOIL_COLORS = {"Clay": "#5B8C3A", "Sand": "#E5B74A", "other": "#9E9E9E", "missing": "#D0D0D0"}

S = U.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
O, PRIV = U.out_dir(S, "01_eda"), U.out_dir(S, "01_eda", private=True)

## 2. Load the Dataset

In [ ]:
# The raw file, all regions and depths
df = U.load_raw(S)
print(f"Dataset shape: {df.shape[0]} rows (sample records), {df.shape[1] - 2} columns")
df.head()

### Column Descriptions
Soil parameter columns of the file. `(calculated)` = derived in the lab sheet from other columns.

In [ ]:
desc = pd.DataFrame([{"column": c, "symbol": s, "unit": u, "meaning": m} for c, (s, u, m) in U.LAB_COLUMNS.items() if c in df])
desc

## 3. Dataset Overview

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary of the soil parameter columns
df[[c for c in U.LAB_COLUMNS if c in df]].describe().T.round(2)

In [ ]:
# Boreholes, regions and sample types
print("Boreholes:", df.BH_Code.nunique(), "| projects:", df.Project_No.nunique() if "Project_No" in df else "-")
display(df.groupby("Region", dropna=False).agg(records=("BH_Code", "size"), boreholes=("BH_Code", "nunique")))
display(df.Sample_Type.value_counts(dropna=False).rename_axis("Sample_Type").to_frame("records"))

**Key observations:** the models use the **Downtown** boreholes only (as V3). Lab tests come mainly from Shelby tubes (`ST`, undisturbed) and SPT-N from split-spoon samples (`SS`), so a record usually has either lab values or an SPT-N, not both.

## 4. Missing Values Analysis

In [ ]:
# Count and percentage of missing values
miss = U.missing_report(df.drop(columns=["mid_depth_m", "record_length_m"]))
miss.to_csv(O / "missing_values.csv")
miss = miss[miss["Missing Count"] > 0]
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(miss.index, miss["Missing %"], color="#4c72b0")
ax.invert_yaxis(); ax.set_xlabel("Missing (%)"); ax.set_title("Missing values per column (all records)")
plt.tight_layout(); plt.savefig(O / "missing_values.png", dpi=200); plt.show()

**Key observations:** missing lab values are normal: each record is either a lab sample or an SPT test, and not every test is run on every sample.

## 5. The Study Area and Depth Used Later
From here on: **Downtown records with the middle of the record in 0-15 m** (the depth of the V3 classification). The rules of notebook 02 (record ≤ 3 m, value ranges, 3 SD) are **not** applied yet.

In [ ]:
dt = df[(df.Region == S["files"]["region_downtown"]) & df.mid_depth_m.between(S["samples"]["top_m"], S["samples"]["bottom_m"])].copy()
dt["soil_group"] = U.soil_group(dt.USCS)
print(f"Downtown 0-15 m: {len(dt)} records, {dt.BH_Code.nunique()} boreholes")
print(f"Records longer than {S['samples']['max_sample_length_m']:g} m (removed in notebook 02):",
      int((dt.record_length_m > S["samples"]["max_sample_length_m"]).sum()))
PARAMS = [c for c in U.LAB_COLUMNS if c in dt]

## 6. Data Availability: Values per Depth Band and Boreholes per Parameter
The data status at a glance. **Usable** = at least 40 of the boreholes have a value (`settings.toml → [eda]`).

In [ ]:
edges = [e for e in S["eda"]["depth_bands_m"] if e <= S["samples"]["bottom_m"]]
avail = U.availability(dt, PARAMS, edges)
status = U.data_status(dt, PARAMS, S)
avail.to_csv(O / "values_per_depth_band.csv")
colors = status.status.map({"usable": "#2c7a3f", "borderline": "#e5b74a", "too few": "#c0392b"})
fig, axes = plt.subplots(1, 2, figsize=(13, 8), gridspec_kw={"width_ratios": [1.5, 1]})
sns.heatmap(avail, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "values"}, ax=axes[0])
axes[0].set_xlabel("Depth band (m)"); axes[0].set_ylabel(""); axes[0].set_title("(a) Values per depth band", loc="left")
axes[1].barh(status.parameter, status.boreholes, color=colors)
axes[1].axvline(S["eda"]["usable_min_boreholes"], ls="--", color="grey", lw=0.8)
axes[1].invert_yaxis(); axes[1].set_xlabel(f"Boreholes with a value (of {dt.BH_Code.nunique()})")
axes[1].set_title("(b) Boreholes per parameter (green = usable)", loc="left")
plt.tight_layout(); plt.savefig(O / "data_availability.png", dpi=200); plt.show()

## 7. Soil Types (USCS)

In [ ]:
# Logged USCS symbols and the Clay / Sand grouping used as the 4th model input
u = dt.USCS.astype(str).str.strip().str.upper().replace("NAN", "missing")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
u.value_counts().plot(kind="bar", ax=axes[0], color="#55a868"); axes[0].set_title("Logged USCS symbols"); axes[0].set_ylabel("records")
dt.soil_group.value_counts().plot(kind="bar", ax=axes[1], color=[SOIL_COLORS[g] for g in dt.soil_group.value_counts().index])
axes[1].set_title("Clay / Sand groups")
plt.tight_layout(); plt.savefig(O / "soil_types.png", dpi=200); plt.show()
u.value_counts().rename_axis("USCS").to_frame("records")

**Key observations:** almost all samples are **CL** or **SC**, so Clay (CL, CH) / Sand (SC, SM, SP) is the natural soil input. Six USCS groups would leave groups with almost no samples.

## 8. Distributions

In [ ]:
# Histograms of the modelled parameters, Clay and Sand separately
TARGET_COLS = {t: S["targets"][t]["column"] for t in S["cv"]["targets"] if S["targets"][t]["column"] in dt}
n = len(TARGET_COLS); ncol = 4; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow))
for ax, (t, c) in zip(axes.ravel(), TARGET_COLS.items()):
    for g in ("Clay", "Sand"):
        ax.hist(dt.loc[dt.soil_group == g, c].dropna(), bins=30, alpha=0.6, color=SOIL_COLORS[g], label=g)
    ax.set_title(f"{t} ({S['targets'][t]['unit']})")
for ax in axes.ravel()[n:]:
    ax.axis("off")
axes.ravel()[0].legend()
plt.tight_layout(); plt.savefig(O / "distributions.png", dpi=200); plt.show()

**Key observations:** check for skewed parameters (SPT-N, Su, Eu, E′ are learned on a log scale, `log = true` in settings) and for values at the limits of the ranges in `settings.toml`.

## 9. Depth Profiles

In [ ]:
# Each value against the middle depth of its record
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4 * nrow), sharey=True)
for ax, (t, c) in zip(axes.ravel(), TARGET_COLS.items()):
    for g in ("Clay", "Sand"):
        d = dt[dt.soil_group == g]
        ax.scatter(d[c], d.mid_depth_m, s=6, alpha=0.5, color=SOIL_COLORS[g], label=g)
    ax.set_title(t); ax.set_xlabel(S["targets"][t]["unit"])
for ax in axes.ravel()[n:]:
    ax.axis("off")
axes.ravel()[0].invert_yaxis(); axes.ravel()[0].legend()
for ax in axes[:, 0]:
    ax.set_ylabel("depth (m)")
plt.tight_layout(); plt.savefig(O / "depth_profiles.png", dpi=200); plt.show()

## 10. By Soil Type

In [ ]:
# Box plots, Clay vs Sand
long = dt[dt.soil_group.isin(["Clay", "Sand"])].melt(id_vars="soil_group", value_vars=list(TARGET_COLS.values()),
                                                       var_name="column", value_name="value").dropna()
long["parameter"] = long.column.map({c: t for t, c in TARGET_COLS.items()})
g = sns.catplot(data=long, x="soil_group", y="value", col="parameter", kind="box", col_wrap=5, sharey=False,
                height=2.8, palette=SOIL_COLORS, order=["Clay", "Sand"])
g.set_titles("{col_name}"); g.set_xlabels("")
plt.savefig(O / "by_soil_type.png", dpi=200); plt.show()
long.groupby(["parameter", "soil_group"]).value.describe().round(2)

## 11. Plasticity Chart (Casagrande)
Fine soils above the A-line are clays (CL, CH), below it silts (ML, MH). The colour is the **logged** USCS group: points far on the wrong side show logging or lab inconsistencies.

In [ ]:
p = dt.dropna(subset=["LL_pct", "PI_pct"])
fig, ax = plt.subplots(figsize=(7, 6))
for gname, d in p.groupby("soil_group"):
    ax.scatter(d.LL_pct, d.PI_pct, s=8, alpha=0.5, color=SOIL_COLORS.get(gname, "grey"), label=gname)
ll = np.linspace(0, max(80, p.LL_pct.max() + 5), 50)
ax.plot(ll, U.a_line(ll), "k-", lw=1, label="A-line PI = 0.73 (LL − 20)")
ax.axvline(50, color="k", ls=":", lw=1)
ax.set_xlabel("Liquid limit LL (%)"); ax.set_ylabel("Plasticity index PI (%)"); ax.set_xlim(0, ll.max()); ax.set_ylim(0)
ax.legend(); plt.tight_layout(); plt.savefig(O / "plasticity_chart.png", dpi=200); plt.show()
print(f"{len(p)} samples; above the A-line: {(p.PI_pct > U.a_line(p.LL_pct)).mean():.0%}")

## 12. Consistency Checks

In [ ]:
checks = {}
# PI = LL - PL
v = dt.dropna(subset=["LL_pct", "PL_pct", "PI_pct"])
checks["PI differs from LL - PL by > 0.5"] = f"{int(((v.LL_pct - v.PL_pct - v.PI_pct).abs() > 0.5).sum())} of {len(v)}"
# Saturation from phase relations: Sr = w Gs / e0
v = dt.dropna(subset=["Water_Content_pct", "Specific_Gravity", "Void_Ratio"])
sr = v.Water_Content_pct / 100 * v.Specific_Gravity / v.Void_Ratio
checks["w Gs / e0 (saturation) median"] = round(float(sr.median()), 2)
checks["w Gs / e0 outside 0.8-1.1"] = f"{int(((sr < 0.8) | (sr > 1.1)).sum())} of {len(v)}"
# Values outside the ranges in settings.toml
for t, c in TARGET_COLS.items():
    lo, hi = S["targets"][t]["min"], S["targets"][t]["max"]
    checks[f"{t} outside {lo}-{hi}"] = int(((dt[c] < lo) | (dt[c] > hi)).sum())
checks["Gs range"] = f"{dt.Specific_Gravity.min():.2f} - {dt.Specific_Gravity.max():.2f}"
pd.Series(checks, name="result").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sr * 100, v.Saturation_pct, s=6, alpha=0.5)
ax.plot([60, 110], [60, 110], "k--", lw=0.8)
ax.set_xlabel("w·Gs / e0 (%)"); ax.set_ylabel("Saturation in the file (%)")
ax.set_title("Phase relations: calculated vs stored saturation")
plt.tight_layout(); plt.savefig(O / "phase_relations.png", dpi=200); plt.show()

## 13. Correlations
Spearman correlation between parameters; the number of samples with both values is shown below it (few pairs = do not trust the value).

In [ ]:
corr_cols = [c for c in TARGET_COLS.values()] + ["mid_depth_m"]
corr = dt[corr_cols].rename(columns={c: t for t, c in TARGET_COLS.items()} | {"mid_depth_m": "depth"}).corr(method="spearman")
pairs = dt[corr_cols].notna().astype(int)
npairs = pd.DataFrame(pairs.T.values @ pairs.values, index=corr.index, columns=corr.columns)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax)
ax.set_title("Spearman correlation (Downtown, 0-15 m)")
plt.tight_layout(); plt.savefig(O / "correlations.png", dpi=200); plt.show()
npairs

## 14. Spatial Coverage *(private: shows borehole locations)*

In [ ]:
bh = dt.groupby("BH_Code").agg(E=("UTM_Easting_m", "first"), N=("UTM_Northing_m", "first"),
                               lab=("Water_Content_pct", "count"), spt=("SPT_N", "count"))
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, col, title in ((axes[0], "lab", "Water content values per borehole"), (axes[1], "spt", "SPT-N values per borehole")):
    sc = ax.scatter(bh.E, bh.N, c=bh[col], s=25, cmap="viridis")
    ax.set_aspect("equal"); ax.set_title(title); fig.colorbar(sc, ax=ax, fraction=0.04)
plt.tight_layout(); plt.savefig(PRIV / "spatial_coverage.png", dpi=200); plt.show()

## 15. Summary: Data Status per Parameter

In [ ]:
status.to_csv(O / "data_status.csv", index=False)
display(status.round(2))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 02 (data cleaning).")

**Key findings** (fill in from the outputs above, not assumed):
- usable parameters and their number of boreholes:
- soil types:
- consistency:
- parameters not modelled and why (Gs nearly constant; calculated columns; c′ and φ′ not needed for movement):